# Check implementation of Eleveld model

In [ ]:
%load_ext autoreload
%autoreload 2

import logging

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import mstats

from propofol.pkpd import EleveldPatient, EleveldPD, EleveldPK, PKPD_solver

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:

patient = EleveldPatient(35, 70, 170, 'male', opiates=True)
p = PKPD_solver(patient, EleveldPK(patient), EleveldPD(patient))

p.print_model_parameters()
# (p.pk.V3(p.age, p.weight, p.bmi) / p.pk.V3(p.age_ref, p.weight_ref, p.bmi_ref))**0.75
print(f"k12/k21: {p.pk.k12 / p.pk.k21}")  # TODO: double check, but appears correct from comparing k10 to SimTIVA
print(f"k13/k31: {p.pk.k13 / p.pk.k31}")
print(f"Q2: {p.pk.Q2_arterial}")
print(f"Q3: {p.pk.Q3}")

In [ ]:
dose50 = p.find_dose_drug_effect_50()
t = np.linspace(0, 15, 15 * 60)
A1, A2, A3, Ce = p(t, [dose50, 0, 0, 0])

plt.plot(t, A1 / p.pk.V1, label='Cp (mcg / mL)')
plt.plot(t, Ce, label='Ce (mcg / mL)')
plt.plot(t, p.pd.Ce50 * np.ones(len(t)), 'k:', label='Ce50')
plt.xlabel('Time (minutes)')
plt.ylabel('Concentration (mcg / ml)')
plt.legend(loc=1)
plt.show()

bis = np.array([p.pd.bis(x) for x in Ce])
plt.plot(t, bis)
logger.info(f"Dose 50% drug effect: {dose50 / p.patient.weight:.2f} mg / kg")
logger.info(f"BIS min / BIS baseline: {bis.min() / p.pd.bis_baseline:.3f}")

#### Figure 5a
We recreate figure 5a of the paper

In [ ]:
age, weight, height = np.array([[0.003, 3.5, 50], [1, 10, 70], [3, 16, 95], [5, 18, 109], [10, 32, 139], [18, 70, 170], [35, 70, 170], [70, 70, 170]]).T
t = np.linspace(0, 15, 15 * 60)

for sex in ['male', 'female']:
    doses = []
    for a, w, h in zip(age, weight, height):

        patient = EleveldPatient(a, w, h, sex, opiates=True)
        p = PKPD_solver(patient, EleveldPK(patient), EleveldPD(patient))
        dose50 = p.find_dose_drug_effect_50()
        doses.append(dose50 / w)
        _, _, _, Ce = p(t, [dose50, 0, 0, 0])
        bis = np.array([p.pd.bis(x) for x in Ce])
        # logger.info(f"--- age: {a}, height: {h}, weight: {w} ---")
        # logger.info(f"Dose 50: {dose50:.3f}")
        # logger.info(f"BIS min: {bis.min()}; BIS baseline: {p.pd.bis_baseline:.3f}")
        # logger.info(f"BIS min / BIS baseline: {bis.min() / p.pd.bis_baseline:.3f}")
    plt.scatter(age, doses, label=sex)
    plt.fill_between([3, 16], [2.5, 2.5], [3.5, 3.5], color='green', alpha=0.4, zorder=-10)  # adults
    plt.fill_between([18, 55], [2, 2], [2.5, 2.5], color='green', alpha=0.4, zorder=-10)  # adults
    plt.fill_between([55, 70], [1, 1], [1.5, 1.5], color='green', alpha=0.4, zorder=-10)  # Elderly
    plt.legend(loc=1)
    plt.ylim(0, 4.7)
    plt.xlim(0, 90)
    plt.xlabel("Age (years)")
    plt.ylabel("Dose for 50% drug effect [mg / kg]")

## Individual variation
Simulation of BIS with between subject variability.

In [ ]:

bis_list = []
t = np.linspace(0, 15, 15 * 60)

# 10k samples takes about 11s
n_samples = 1000
for i in range(n_samples):
    if i == 0:

        pk = EleveldPK(patient, use_bsv=False)
        pd = EleveldPD(patient, use_bsv=False)
        p = PKPD_solver(patient, pk, pd)
        dose50 = p.find_dose_drug_effect_50()
    else:
        p.pk.draw_eta()
        p.pd.draw_eta()
    A1, A2, A3, Ce = p(t, [dose50, 0, 0, 0])
    bis = np.array([p.pd.bis(x) for x in Ce])
    bis_list.append(bis)
bis_list = np.array(bis_list)
bis_lo, bis_hi = mstats.mquantiles(bis_list, [0.025, 0.975], axis=0)
for ix, bis in enumerate(bis_list):
    if ix == 0:
        plt.plot(t, bis, 'r-', zorder=10, label=f"95% C.I.")
        plt.plot(t, bis_lo, 'r:', zorder=10)
        plt.plot(t, bis_hi, 'r:', zorder=10)
    else:
        plt.plot(t, bis, color='k', alpha=0.01)
plt.legend(loc=4)
plt.show()
